# **Task VIII: Vision Transformer (ViT) & Quantum Vision Transformer (QViT)**

---

## **1. Executive Summary**
This task explores the shift from traditional Convolutional architectures to **Transformer-based models** for computer vision. While CNNs rely on local inductive biases, the **Vision Transformer (ViT)** captures global dependencies by treating an image as a sequence of patches.

Our objective is twofold:
1.  **Classical Baseline:** Implementing a ViT on the **MNIST dataset** to establish a performance benchmark.
2.  **Quantum Future:** Proposing a detailed architecture for a **Quantum Vision Transformer (QViT)** that utilizes Hilbert space dimensionality and entanglement for feature extraction.



---

## **2. Core Concept: Self-Attention Mechanism**
The mathematical heart of this notebook is the **Multi-Head Self-Attention (MHSA)**. It allows the model to dynamically weight the importance of different image patches:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In the proposed **Quantum Extension**, we aim to replace this classical dot-product with **Quantum Fidelity** measurements ($|\langle \psi | \phi \rangle|^2$) using a SWAP Test circuit.



---

## **3. Technical Comparison at a Glance**

| Feature | Classical ViT | Quantum ViT (Proposed) |
| :--- | :--- | :--- |
| **Data Space** | Euclidean ($\mathbb{R}^d$) | Hilbert ($2^n$) |
| **Similarity** | Dot Product | Quantum Overlap |
| **Correlation** | Attention Weights | Quantum Entanglement |

## **Setup and imports**

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader,Subset
from tqdm import tqdm

**VIT Class Structure**

In [4]:
# ViT Components
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_chans=1, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x) # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)
        return x

class SimpleViT(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, num_layers=2, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding()
        self.pos_embed = nn.Parameter(torch.zeros(1, 16, embed_dim)) # 16 patches

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.cls_head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.cls_head(x)


Getting Data of **MNIST** Dataset

In [29]:
from logging import root
transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize()
])
train_data = torchvision.datasets.MNIST(
    root = './content',
    train = True,
    transform = transform,
    download = True
)
test_data = torchvision.datasets.MNIST(
    root = './content',
    train = False,
    transform = transform,
    download = True
)
# subset_indices = list(range(50000))
# train_data = Subset(train_data, subset_indices)
# test_data = Subset(test_data, subset_indices)


train_data_loader = DataLoader(
 train_data,
 shuffle=True,
 num_workers=1,
 batch_size=90
)
test_data_loader = DataLoader(
 test_data,
 shuffle=True,
 num_workers=1,
 batch_size=90
)

**Training setup**

In [33]:
# Training Logic
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleViT().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train(epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data, target in tqdm(train_data_loader, desc=f"Epoch {epoch+1}"):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Average Loss: {total_loss/len(train_data_loader):.4f}")

# Execute
train(epochs=2)


/tmp/ipykernel_251/2264102341.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 1: 100%|██████████| 667/667 [03:38<00:00,  3.06it/s]


Epoch 1 Average Loss: 0.7527


Epoch 2: 100%|██████████| 667/667 [03:41<00:00,  3.02it/s]

Epoch 2 Average Loss: 0.1824


**Evaluation on test data**

## Training Progress

The model has currently been trained for only **2 epochs** due to high computational and training time.

- Current accuracy after 2 epochs: **96.45%**
- Expected accuracy after 4–5 epochs: **98–99%**

With additional training, the model is expected to converge to higher accuracy.

In [36]:
def evaluate():
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = 100. * correct / len(test_data_loader.dataset)
    print(f"\nFinal Test Accuracy: {accuracy:.2f}%")



evaluate()


Final Test Accuracy: 96.45%


# **Part 2: Quantum Vision Transformer (QViT) Architecture**

---

## **1. Conceptual Extension: From Classical to Quantum**
The transition from a Classical Vision Transformer (ViT) to a **Quantum Vision Transformer (QViT)** involves replacing the most computationally intensive components—Self-Attention and Feed-Forward Networks—with Quantum circuits that operate in a high-dimensional Hilbert space.

### **Key Quantum Enhancements:**
* **Quantum Feature Maps:** These maps encode classical patch data into quantum states using techniques like Angle or Amplitude encoding.
* **Quantum Attention:** This mechanism computes attention scores via quantum circuits or kernels to measure state-to-state similarities.
* **Quantum Kernels:** These are used to measure the overlap between representations, often more efficiently than classical dot products in high dimensions.
* **Hybrid Integration:** A hybrid architecture combines quantum layers for similarity measurement with classical layers for optimization and final classification.



---

## **2. Detailed QViT Architecture Sketch**

The proposed QViT follows a sequential hybrid pipeline designed to leverage quantum entanglement for capturing complex correlations in image data.

### **Stage 1: Classical Patch Embedding**
* **Input:** Raw $28 \times 28$ MNIST images.
* **Decomposition:** Images are divided into non-overlapping patches (e.g., $4 \times 4$ pixels).
* **Dimensionality Reduction:** Classical embeddings (often 192-dim) are reduced using PCA or linear layers to match the available qubit count (e.g., 8 qubits).

### **Stage 2: Quantum State Preparation (Encoding)**
* **Method:** **Angle Encoding** is used to map the normalized patch features onto the Bloch sphere.
* **Circuit Logic:** Each feature $x_i$ is mapped to a rotation gate angle $\theta_i$ (e.g., $RY(x_i)$ and $RZ(x_i)$) to prepare the initial state $|\psi\rangle$.



### **Stage 3: Quantum Attention Mechanism**


Instead of the classical dot-product $Attention(Q, K, V) = \text{softmax}(QK^T/\sqrt{d_k})V$, the QViT uses:
* **Fidelity Measurement:** A **SWAP Test** circuit computes the overlap $|\langle \psi_1 | \psi_2 \rangle|^2$ between patch representations.
* **Entanglement Layers:** Parameterized CNOT chains create correlations between qubits representing different patches, mimicking the "Global Attention" of classical transformers.
* **Variational Layers:** Parameterized Quantum Circuits (PQC) with trainable weights $\theta$ act as the learning parameters for the transformer encoder.

### **Stage 4: Measurement and Classification Head**
* **Readout:** Perform Pauli-Z expectation value measurements to extract information from the quantum state.
* **Classical MLP:** The measured values are fed into a classical multi-layer perceptron (MLP) for the final 10-digit classification.



---

## **3. Structural Comparison**

| Feature | Classical ViT (Implemented) | Quantum ViT (Proposed) |
| :--- | :--- | :--- |
| **Data Representation** | Dense Vectors ($R^d$) | Quantum States ($2^n$ Hilbert Space) |
| **Similarity Metric** | Dot Product ($QK^T$) | Quantum Fidelity / Overlap |
| **Parameters** | Millions of float32 weights | Variational Gate Angles ($\theta$) |
| **Primary Advantage** | High accuracy on large data | Exponential feature space and entanglement |

---

## **4. Conclusion**
The QViT extension offers a promising path for High Energy Physics (HEP) analysis by utilizing the natural ability of quantum circuits to represent and correlate complex particle data. By using **NISQ-friendly** designs like hybrid encoding and shallow variational circuits, this architecture remains viable for current quantum hardware while providing a foundation for future quantum advantage.

# **Architecture Sketch of QViT**

```
                    +----------------------+
                    |      INPUT IMAGE     |
                    |        28 × 28       |
                    +----------------------+
                               |
                               v
                    +---------------------------+
                    |     CLASSICAL PATCHING    |
                    |  Split into 16 patches    |
                    |      (each 7 × 7)         |
                    +---------------------------+
                               |
                               v
                    +----------------------------------+
                    |  LINEAR PROJECTION + PCA         |
                    |  49 pixels → feature vector      |
                    |  192 features → 8 dimensions     |
                    |  (mapped to 8 qubits)            |
                    +----------------------------------+
                               |
                               v
        +---------------------------+      +---------------------------+
        |   QUANTUM ENCODING U(x)   |      |   QUANTUM ENCODING U(x)   |
        |   Angle Encoding:         |      |   Angle Encoding:         |
        |   RY(x_i), RZ(x_i)        | ...  |   RY(x_i), RZ(x_i)        |
        |   (Patch Token 1)         |      |   (Patch Token N)         |
        +---------------------------+      +---------------------------+
                     \                               /
                      \                             /
                       +------------+---------------+
                                    |
                                    v
                    +--------------------------------------+
                    |       QUANTUM SELF-ATTENTION         |
                    |--------------------------------------|
                    |  1. CNOT Entangling Chains           |
                    |     (Global correlations)             |
                    |  2. SWAP Test (Fidelity Scores)      |
                    |     A_ij = |<ψ_i | ψ_j>|^2            |
                    |  3. Variational Rotations (θ)         |
                    |     Trainable parameters              |
                    +--------------------------------------+
                                    |
                                    v
                    +--------------------------------------+
                    |        MEASUREMENT AND READOUT       |
                    |  Pauli-Z expectation values <Z>      |
                    |  Classical feature vector output     |
                    +--------------------------------------+
                                    |
                                    v
                    +--------------------------------------+
                    |          CLASSICAL MLP HEAD          |
                    |  Linear → Activation → Linear        |
                    |  Output: 10-class probabilities      |
                    +--------------------------------------+
                                    |
                                    v
                        +--------------------------+
                        |  FINAL PREDICTION        |
                        |      Example: Digit 5    |
                        +--------------------------+
```

# **Quantum Attention Block**

```
                    QUBIT REGISTER A                     QUBIT REGISTER B
                  (Patch Embedding i)                  (Patch Embedding j)
                           |                                   |
                +-----------------------+           +-----------------------+
                |     STATE PREP U(x_i) |           |     STATE PREP U(x_j) |
                |   |ψ_i⟩ = U(x_i)|0⟩   |           |   |ψ_j⟩ = U(x_j)|0⟩   |
                +-----------------------+           +-----------------------+
                           |                                   |
                           +---------------+-------------------+
                                           |
                                           v
                        +---------------------------------------+
                        |          SWAP TEST CIRCUIT            |
                        |---------------------------------------|
                        |  Ancilla:  |0⟩ ── H ──●── H ── Measure|
                        |                       │               |
                        |  Register A: |ψ_i⟩ ───X────────────── |
                        |                       │               |
                        |  Register B: |ψ_j⟩ ───X────────────── |
                        +---------------------------------------+
                                           |
                                           v
                        +---------------------------------------+
                        |  Fidelity Computation                  |
                        |  P(0) = (1 + |⟨ψ_i | ψ_j⟩|²) / 2        |
                        |  F_ij = |⟨ψ_i | ψ_j⟩|²                   |
                        +---------------------------------------+
                                           |
                                           v
                        +---------------------------------------+
                        |  Attention Weight A_ij = F_ij         |
                        |  Used in Quantum Self-Attention       |
                        +---------------------------------------+
```

## **4. Comparative Analysis: Classical ViT vs. Quantum ViT**

The following table summarizes the key architectural and computational differences between the implemented **Classical Vision Transformer** and the proposed **Quantum Extension (QViT)**.

| Feature | Classical Vision Transformer (ViT) | Quantum Vision Transformer (QViT) |
| :--- | :--- | :--- |
| **Data Representation** | Dense Vectors in Euclidean space ($\mathbb{R}^d$). | Quantum States in Hilbert space ($2^n$ dimensions). |
| **Feature Extraction** | Linear Projections & MLP layers. | Variational Quantum Circuits (VQC) with Data Re-uploading. |
| **Attention Mechanism** | Scaled Dot-Product: $\text{softmax}(QK^T/\sqrt{d_k})V$. | **Quantum Fidelity / Overlap** measured via SWAP Test. |
| **Correlation Learning** | Multi-Head Self-Attention weights. | **Quantum Entanglement** (CNOT/CZ Chains). |
| **Parameter Efficiency** | Millions of trainable weights (High memory overhead). | Variational Gate Angles ($\theta$) (Extremely compact). |
| **Expressivity** | Polynomial growth with dimension. | Exponential growth (Hilbert space expansion). |
| **Optimization** | Gradient Descent on classical Loss surface. | Hybrid optimization using **Parameter-Shift rules**. |
| **Hardware Goal** | High-performance GPUs/TPUs. | NISQ-era Quantum Processors (e.g., IBM, IonQ). |

---

###**Technical Summary:**
The primary difference lies in how the models perceive "similarity." While the **Classical ViT** relies on numerical dot-products, the **QViT** utilizes the physical properties of quantum states. By mapping patches to qubits, we can use **Entanglement** to capture global dependencies in a single quantum step, which is a significant theoretical improvement for complex datasets like High Energy Physics jets.